<a href="https://colab.research.google.com/github/ahmed-lay/fullstack-expense-tracker/blob/main/emo_det_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DataCollatorWithPadding,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
import os

# Disable wandb and other logging
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["HF_EVALUATE_OFFLINE"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

print("=" * 60)
print("BALANCED EMOTION CLASSIFICATION WITH DISTILBERT")
print("=" * 60)

BALANCED EMOTION CLASSIFICATION WITH DISTILBERT


### STEP 1: LOAD AND FILTER DATA

In [ ]:
print("\n[1/7] Loading dataset...")

# Load your dataset
df = pd.read_csv("/content/emotions_dataset.csv")
df = df.rename(columns={"Clean_Text": "Sentence", "Emotion": "Label"})
df = df.dropna(subset=["Sentence", "Label"])

print(f"Total samples loaded: {len(df)}")
print(f"\nOriginal class distribution:")
print(df["Label"].value_counts())

# Filter to keep only 6 emotions
TARGET_EMOTIONS = ['happiness', 'sadness', 'neutral', 'anger', 'love', 'fear']
df_filtered = df[df["Label"].isin(TARGET_EMOTIONS)].copy()

print(f"\n[2/7] Filtered to 6 emotions...")
print(f"Samples after filtering: {len(df_filtered)}")
print(f"\nFiltered class distribution:")
print(df_filtered["Label"].value_counts())


[1/7] Loading dataset...
Total samples loaded: 131306

Original class distribution:
Label
happiness    31205
sadness      17809
neutral      15733
anger        13341
love         10512
fear          8795
disgust       8407
confusion     8209
surprise      4560
shame         4248
guilt         3470
sarcasm       2534
desire        2483
Name: count, dtype: int64

[2/7] Filtered to 6 emotions...
Samples after filtering: 97395

Filtered class distribution:
Label
happiness    31205
sadness      17809
neutral      15733
anger        13341
love         10512
fear          8795
Name: count, dtype: int64


### STEP 2: BALANCE THE DATASET

In [ ]:
print(f"\n[3/7] Balancing dataset...")

# Find the minimum class size
min_class_size = df_filtered["Label"].value_counts().min()
print(f"Minimum class size: {min_class_size}")

# Option 1: Use minimum class size (most balanced but loses data)
# samples_per_class = min_class_size

# Option 2: Use a reasonable number (compromise between balance and data)
samples_per_class = 15000  # Slightly less than fear (7028)
print(f"Using {samples_per_class} samples per class")

# Undersample each class
balanced_dfs = []
for emotion in TARGET_EMOTIONS:
    emotion_df = df_filtered[df_filtered["Label"] == emotion]
    if len(emotion_df) >= samples_per_class:
        sampled_df = emotion_df.sample(n=samples_per_class, random_state=42)
    else:
        sampled_df = emotion_df  # Use all if less than target
    balanced_dfs.append(sampled_df)

df_balanced = pd.concat(balanced_dfs, ignore_index=True)

# Shuffle the balanced dataset
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset size: {len(df_balanced)}")
print(f"\nBalanced class distribution:")
print(df_balanced["Label"].value_counts())


[3/7] Balancing dataset...
Minimum class size: 8795
Using 15000 samples per class

Balanced dataset size: 77648

Balanced class distribution:
Label
sadness      15000
neutral      15000
happiness    15000
anger        13341
love         10512
fear          8795
Name: count, dtype: int64


### STEP 3: ENCODE LABELS

In [ ]:
print(f"\n[4/7] Encoding labels...")

labels_list = sorted(df_balanced["Label"].unique().tolist())
label2id = {label: i for i, label in enumerate(labels_list)}
id2label = {i: label for label, i in label2id.items()}

df_balanced["Label"] = df_balanced["Label"].map(label2id)

print(f"\nLabel mapping:")
for label, idx in label2id.items():
    print(f"  {label}: {idx}")


[4/7] Encoding labels...

Label mapping:
  anger: 0
  fear: 1
  happiness: 2
  love: 3
  neutral: 4
  sadness: 5


### STEP 4: SPLIT DATA

In [ ]:
print(f"\n[5/7] Splitting data...")

train_df, test_df = train_test_split(
    df_balanced,
    test_size=0.2,
    stratify=df_balanced["Label"],
    random_state=42
)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[["Sentence", "Label"]])
test_dataset = Dataset.from_pandas(test_df[["Sentence", "Label"]])


[5/7] Splitting data...
Training samples: 62118
Test samples: 15530


### STEP 5: TOKENIZE DATA

In [ ]:
print(f"\n[6/7] Loading tokenizer and tokenizing...")

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["Sentence"],
        truncation=True,
        padding=False,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Rename Label to labels (required by Trainer)
train_dataset = train_dataset.rename_column("Label", "labels")
test_dataset = test_dataset.rename_column("Label", "labels")

# Create data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


[6/7] Loading tokenizer and tokenizing...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Map:   0%|          | 0/62118 [00:00<?, ? examples/s]

Map:   0%|          | 0/15530 [00:00<?, ? examples/s]

### STEP 6: INITIALIZE MODEL

In [ ]:
print(f"\n[7/7] Initializing model...")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels_list),
    id2label=id2label,
    label2id=label2id
)


[7/7] Initializing model...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### STEP 7: TRAINING SETUP

In [ ]:
print("\n" + "=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="balanced_emotion_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    save_total_limit=2  # Only keep 2 best checkpoints
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


TRAINING CONFIGURATION


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-3789965691.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### STEP 8: TRAIN MODEL

In [ ]:
print("\nStarting training...")
print("This will take approximately 30-40 minutes on GPU\n")

trainer.train()


Starting training...
This will take approximately 30-40 minutes on GPU



Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.604500,0.581132,0.796909,0.796979
2,0.481800,0.567604,0.801803,0.801611
3,0.373800,0.608103,0.799227,0.799391
4,0.290500,0.684154,0.798905,0.798911
5,0.217600,0.766989,0.794784,0.794837


TrainOutput(global_step=19415, training_loss=0.4216663546271156, metrics={'train_runtime': 1484.734, 'train_samples_per_second': 209.189, 'train_steps_per_second': 13.076, 'total_flos': 3129072103522080.0, 'train_loss': 0.4216663546271156, 'epoch': 5.0})

### STEP 9: EVALUATE MODEL

In [ ]:
print("\n" + "=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

eval_results = trainer.evaluate()
print(f"\nFinal Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"Final F1 Score: {eval_results['eval_f1']:.4f}")

# Get detailed classification report
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("\nDetailed Classification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=labels_list,
    digits=4
))


FINAL EVALUATION



Final Accuracy: 0.8018
Final F1 Score: 0.8016

Detailed Classification Report:
              precision    recall  f1-score   support

       anger     0.7496    0.8426    0.7934      2668
        fear     0.8476    0.8317    0.8396      1759
   happiness     0.8283    0.7170    0.7686      3000
        love     0.7481    0.8388    0.7909      2103
     neutral     0.7878    0.7637    0.7756      3000
     sadness     0.8617    0.8450    0.8532      3000

    accuracy                         0.8018     15530
   macro avg     0.8038    0.8065    0.8035     15530
weighted avg     0.8047    0.8018    0.8016     15530



### STEP 10: SAVE MODEL

In [ ]:
print("\n" + "=" * 60)
print("SAVING MODEL")
print("=" * 60)

save_path = "distilbert_balanced_emotion_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"\nModel saved to: {save_path}")

# Save label mappings
import json
with open(f"{save_path}/label_mappings.json", "w") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, indent=2)

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)


SAVING MODEL

Model saved to: distilbert_balanced_emotion_model

TRAINING COMPLETE!


### STEP 11: TEST PREDICTION FUNCTION

In [ ]:
print("\nTesting prediction function...")

def predict_emotion(text, model, tokenizer, id2label):
    """Predict emotion for a given text"""
    device = next(model.parameters()).device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    pred_id = torch.argmax(logits, dim=1).item()
    probabilities = torch.softmax(logits, dim=1)[0]

    return {
        "emotion": id2label[pred_id],
        "confidence": probabilities[pred_id].item(),
        "all_probabilities": {
            id2label[i]: prob.item()
            for i, prob in enumerate(probabilities)
        }
    }

# Test examples
test_texts = [
    "I am so happy and excited about this!",
    "This makes me really sad and depressed",
    "I am furious about what happened!",
    "I love spending time with my family",
    "I'm scared of what might happen",
    "The weather is okay today"
]

print("\nTest Predictions:")
print("-" * 60)
for text in test_texts:
    result = predict_emotion(text, model, tokenizer, id2label)
    print(f"\nText: {text}")
    print(f"Predicted: {result['emotion']} ({result['confidence']:.3f})")
    print(f"Top 3 emotions:")
    sorted_probs = sorted(
        result['all_probabilities'].items(),
        key=lambda x: x[1],
        reverse=True
    )[:3]
    for emotion, prob in sorted_probs:
        print(f"  - {emotion}: {prob:.3f}")

print("\n" + "=" * 60)
print("ALL DONE! Model ready to use.")
print("=" * 60)

# Instructions for downloading
print("\n📥 TO DOWNLOAD YOUR MODEL:")
print("Run these commands in a new cell:")
print("\n!zip -r distilbert_balanced_emotion_model.zip distilbert_balanced_emotion_model")
print("from google.colab import files")
print("files.download('distilbert_balanced_emotion_model.zip')")


Testing prediction function...

Test Predictions:
------------------------------------------------------------

Text: I am so happy and excited about this!
Predicted: happiness (0.994)
Top 3 emotions:
  - happiness: 0.994
  - love: 0.004
  - neutral: 0.001

Text: This makes me really sad and depressed
Predicted: sadness (0.989)
Top 3 emotions:
  - sadness: 0.989
  - anger: 0.005
  - fear: 0.003

Text: I am furious about what happened!
Predicted: anger (0.990)
Top 3 emotions:
  - anger: 0.990
  - happiness: 0.007
  - neutral: 0.001

Text: I love spending time with my family
Predicted: love (0.864)
Top 3 emotions:
  - love: 0.864
  - happiness: 0.123
  - neutral: 0.007

Text: I'm scared of what might happen
Predicted: fear (0.991)
Top 3 emotions:
  - fear: 0.991
  - sadness: 0.005
  - anger: 0.002

Text: The weather is okay today
Predicted: neutral (0.659)
Top 3 emotions:
  - neutral: 0.659
  - happiness: 0.249
  - love: 0.056

ALL DONE! Model ready to use.

📥 TO DOWNLOAD YOUR MODEL:
Ru

  adding: distilbert_balanced_emotion_model/ (stored 0%)
  adding: distilbert_balanced_emotion_model/special_tokens_map.json (deflated 42%)
  adding: distilbert_balanced_emotion_model/label_mappings.json (deflated 50%)
  adding: distilbert_balanced_emotion_model/tokenizer_config.json (deflated 75%)
  adding: distilbert_balanced_emotion_model/training_args.bin (deflated 53%)
  adding: distilbert_balanced_emotion_model/config.json (deflated 50%)
  adding: distilbert_balanced_emotion_model/vocab.txt (deflated 53%)
  adding: distilbert_balanced_emotion_model/model.safetensors (deflated 8%)
  adding: distilbert_balanced_emotion_model/tokenizer.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>